<a href="https://colab.research.google.com/github/haya-web/The-ICoder/blob/main/AR_GNN_XAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()   # choose colab_setup_all_files.py (the 54,149-byte version)

Saving colab_setup_all_files (2).py to colab_setup_all_files (2).py


In [2]:
%run colab_setup_all_files.py
!pip install torch_geometric requests -q
!ls -la *.py

Exception: File `'colab_setup_all_files.py'` not found.

In [3]:
  !unzip -o AR-GNN-XAI-complete.zip
  %cd AR-GNN-XAI-complete/code
  %run colab_setup_all_files.py

unzip:  cannot find or open AR-GNN-XAI-complete.zip, AR-GNN-XAI-complete.zip.zip or AR-GNN-XAI-complete.zip.ZIP.
[Errno 2] No such file or directory: 'AR-GNN-XAI-complete/code'
/content


Exception: File `'colab_setup_all_files.py'` not found.

In [4]:
   from google.colab import files
   uploaded = files.upload()

Saving colab_setup_all_files (2).py to colab_setup_all_files (2) (1).py


In [5]:
!mv "colab_setup_all_files (2) (1).py" colab_setup_all_files.py
%run colab_setup_all_files.py
!pip install torch_geometric requests -q
!ls -la *.py

Wrote: ['baselines.py', 'data_loader.py', 'evaluate.py', 'explain.py', 'fetch_openalex.py', 'generate_results.py', 'model.py', 'train.py']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.2 MB/s eta 0:00:00
-rw-r--r-- 1 root root  3940 Oct 17 12:00  baselines.py
-rw-r--r-- 1 root root 54253 Oct 17 11:55 'colab_setup_all_files (2).py'
-rw-r--r-- 1 root root 54253 Oct 17 11:59  colab_setup_all_files.py
-rw-r--r-- 1 root root  7534 Oct 17 12:00  data_loader.py
-rw-r--r-- 1 root root  3498 Oct 17 12:00  evaluate.py
-rw-r--r-- 1 root root  3385 Oct 17 12:00  explain.py
-rw-r--r-- 1 root root  3837 Oct 17 12:00  fetch_openalex.py
-rw-r--r-- 1 root root 10293 Oct 17 12:00  generate_results.py
-rw-r--r-- 1 root root  3597 Oct 17 12:00  model.py
-rw-r--r-- 1 root root  4311 Oct 17 12:00  train.py


In [6]:
!grep -c "_ScoreOnlyWrapper" explain.py
!grep -c "manual_seed" train.py
import torch
print("CUDA available:", torch.cuda.is_available())

2
1
CUDA available: True


In [7]:
!python fetch_openalex.py --concept "graph neural network" \
    --from-year 2018 --to-year 2025 --max-papers 8000 \
    --out authors_papers.csv --email you@example.com

Fetching up to 8000 papers on 'graph neural network' (2018-2025)...
  fetched 200 papers so far...
Traceback (most recent call last):
  File "/content/fetch_openalex.py", line 105, in <module>
    papers = fetch_papers(args.concept, args.from_year, args.to_year,
                           args.max_papers, args.email)
  File "/content/fetch_openalex.py", line 50, in fetch_papers
    r.raise_for_status()
    ~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 401 Client Error: Unauthorized for url: https://api.openalex.org/works?search=graph+neural+network&filter=from_publication_date%3A2018-01-01%2Cto_publication_date%3A2025-12-31&per-page=200&mailto=you%40example.com&cursor=IlsxODcuMTA1MjIsIDE2NDA5OTUyMDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMzIxNTYwMjk1NyddIg%3D%3D


In [8]:
OPENALEX_KEY = "1kKA1uUOM22Ax6Pc1sc15k"

In [9]:
%%writefile fetch_openalex.py
import argparse, csv, time, requests

BASE = "https://api.openalex.org/works"


def fetch_papers(concept, from_year, to_year, max_papers, email, api_key=None):
    papers = []
    cursor = "*"
    params_base = {
        "search": concept,
        "filter": f"from_publication_date:{from_year}-01-01,"
                  f"to_publication_date:{to_year}-12-31",
        "per-page": 100,
        "mailto": email,
    }
    if api_key:
        params_base["api_key"] = api_key
    while len(papers) < max_papers:
        params = dict(params_base, cursor=cursor)
        r = requests.get(BASE, params=params, timeout=30)
        if r.status_code in (401, 403):
            raise SystemExit(f"OpenAlex returned {r.status_code}: check that your API key is correct and activated.")
        r.raise_for_status()
        data = r.json()
        results = data.get("results", [])
        if not results:
            break
        papers.extend(results)
        cursor = data.get("meta", {}).get("next_cursor")
        print(f"  fetched {len(papers)} papers so far...")
        if not cursor:
            break
        time.sleep(0.1)
    return papers[:max_papers]


def write_csv(papers, out_path):
    rows = 0
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["paper_id", "author_id", "author_position",
                          "n_authors", "citation_count"])
        for p in papers:
            pid = p.get("id", "")
            citation_count = p.get("cited_by_count", 0)
            authorships = p.get("authorships", [])
            n_authors = len(authorships)
            if n_authors == 0:
                continue
            for pos, a in enumerate(authorships, start=1):
                author_id = a.get("author", {}).get("id", "")
                if not author_id:
                    continue
                writer.writerow([pid, author_id, pos, n_authors, citation_count])
                rows += 1
    print(f"Wrote {rows} author-paper rows to {out_path}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--concept", required=True)
    ap.add_argument("--from-year", type=int, default=2015)
    ap.add_argument("--to-year", type=int, default=2025)
    ap.add_argument("--max-papers", type=int, default=15000)
    ap.add_argument("--out", default="authors_papers.csv")
    ap.add_argument("--email", default="you@example.com")
    ap.add_argument("--api-key", default=None)
    args = ap.parse_args()
    print(f"Fetching up to {args.max_papers} papers on '{args.concept}' ({args.from_year}-{args.to_year})...")
    papers = fetch_papers(args.concept, args.from_year, args.to_year,
                          args.max_papers, args.email, args.api_key)
    write_csv(papers, args.out)

Overwriting fetch_openalex.py


In [10]:
!python fetch_openalex.py --api-key $OPENALEX_KEY --concept "graph neural network" --from-year 2018 --to-year 2025 --max-papers 8000 --out authors_papers.csv --email mamoonaabid111@gmail.com

Fetching up to 8000 papers on 'graph neural network' (2018-2025)...
  fetched 100 papers so far...
  fetched 200 papers so far...
  fetched 300 papers so far...
  fetched 400 papers so far...
  fetched 500 papers so far...
  fetched 600 papers so far...
  fetched 700 papers so far...
  fetched 800 papers so far...
  fetched 900 papers so far...
  fetched 1000 papers so far...
  fetched 1100 papers so far...
  fetched 1200 papers so far...
  fetched 1300 papers so far...
  fetched 1400 papers so far...
  fetched 1500 papers so far...
  fetched 1600 papers so far...
  fetched 1700 papers so far...
  fetched 1800 papers so far...
  fetched 1900 papers so far...
  fetched 2000 papers so far...
  fetched 2100 papers so far...
  fetched 2200 papers so far...
  fetched 2300 papers so far...
  fetched 2400 papers so far...
  fetched 2500 papers so far...
  fetched 2600 papers so far...
  fetched 2700 papers so far...
  fetched 2800 papers so far...
  fetched 2900 papers so far...
  fetched 300

In [11]:
!python train.py --data authors_papers.csv --format csv --epochs 200 --out author_scores.csv | tee train_log.txt

Loading data (csv)...
7821 papers, 21684 authors loaded.
Co-authorship graph: 21684 nodes, 75779 edges.
epoch   10 | train_loss 0.0544 | val_loss 0.0326
epoch   20 | train_loss 0.0425 | val_loss 0.0298
epoch   30 | train_loss 0.0432 | val_loss 0.0324
epoch   40 | train_loss 0.0383 | val_loss 0.0293
epoch   50 | train_loss 0.0347 | val_loss 0.0301
epoch   60 | train_loss 0.0354 | val_loss 0.0305
epoch   70 | train_loss 0.0340 | val_loss 0.0281
epoch   80 | train_loss 0.0318 | val_loss 0.0272
epoch   90 | train_loss 0.0336 | val_loss 0.0270
epoch  100 | train_loss 0.0318 | val_loss 0.0231
epoch  110 | train_loss 0.0327 | val_loss 0.0259
epoch  120 | train_loss 0.0308 | val_loss 0.0266
epoch  130 | train_loss 0.0289 | val_loss 0.0246
epoch  140 | train_loss 0.0297 | val_loss 0.0241
epoch  150 | train_loss 0.0307 | val_loss 0.0224
epoch  160 | train_loss 0.0313 | val_loss 0.0214
epoch  170 | train_loss 0.0275 | val_loss 0.0186
epoch  180 | train_loss 0.0288 | val_loss 0.0212
epoch  190 | t

In [12]:
from google.colab import files
!zip -r ar_gnn_xai_backup.zip authors_papers.csv author_scores.csv best_model.pt training_log.csv train_log.txt *.py
files.download("ar_gnn_xai_backup.zip")

  adding: authors_papers.csv (deflated 86%)
  adding: author_scores.csv (deflated 78%)
  adding: best_model.pt (deflated 18%)
  adding: training_log.csv (deflated 50%)
  adding: train_log.txt (deflated 72%)
  adding: baselines.py (deflated 60%)
  adding: colab_setup_all_files (2).py (deflated 64%)
  adding: colab_setup_all_files.py (deflated 64%)
  adding: data_loader.py (deflated 63%)
  adding: evaluate.py (deflated 58%)
  adding: explain.py (deflated 56%)
  adding: fetch_openalex.py (deflated 61%)
  adding: generate_results.py (deflated 67%)
  adding: model.py (deflated 59%)
  adding: train.py (deflated 64%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
!python generate_results.py --data authors_papers.csv --format csv --model best_model.pt --scores author_scores.csv --run_gnnexplainer | tee results_log.txt

Papers: 7821
Authors (nodes): 21684
Co-authorship edges: 75779
Mean degree: 6.99
Median degree: 4.00
Max degree: 218
Isolated/degree-1 authors (novices): 2046 (9.4%)


== GNN (AR-GNN-XAI) ==
  vs h_index: {'spearman': 0.5878, 'kendall': 0.4763, 'ndcg@20': 0.1108, 'n': 21684}
  vs p_index: {'spearman': 0.9706, 'kendall': 0.8753, 'ndcg@20': 0.8522, 'n': 21684}

== degree ==
  vs h_index: {'spearman': 0.4101, 'kendall': 0.3472, 'ndcg@20': 0.1584, 'n': 21684}
  vs p_index: {'spearman': 0.3633, 'kendall': 0.2647, 'ndcg@20': 0.0043, 'n': 21684}

== closeness ==
  vs h_index: {'spearman': 0.3545, 'kendall': 0.2856, 'ndcg@20': 0.77, 'n': 21684}
  vs p_index: {'spearman': 0.3462, 'kendall': 0.2434, 'ndcg@20': 0.1097, 'n': 21684}

== NAP ==
  vs h_index: {'spearman': 0.6735, 'kendall': 0.5621, 'ndcg@20': 0.4171, 'n': 21684}
  vs p_index: {'spearman': 0.9177, 'kendall': 0.7745, 'ndcg@20': 0.6628, 'n': 21684}
Wrote results/comparison_table.csv

=== STAR AUTHOR: https://openalex.org/A5100349440 (gn

In [14]:
s = open("generate_results.py").read()
old = """    sorted_by_score = sorted(gnn_scores.items(), key=lambda t: -t[1])
    star_author = sorted_by_score[0][0]
    degrees_dict = dict(G.degree())
    novices = [a for a in authors if degrees_dict.get(a, 0) in (1, 2)]
    novice_author = novices[0] if novices else authors[-1]
"""
new = """    degrees_dict = dict(G.degree())
    sorted_by_score = sorted(gnn_scores.items(), key=lambda t: -t[1])
    top20 = [a for a, _ in sorted_by_score[:20]]
    star_author = max(top20, key=lambda a: degrees_dict.get(a, 0))
    deg1 = [a for a in authors
            if degrees_dict.get(a, 0) == 1 and a != star_author and a in gnn_scores]
    if deg1:
        med = float(np.median([gnn_scores[a] for a in deg1]))
        novice_author = min(deg1, key=lambda a: abs(gnn_scores[a] - med))
    else:
        novice_author = next(a for a in authors if a != star_author)
    print(f"Selected hub author: {star_author} (degree {degrees_dict.get(star_author, 0)}); "
          f"novice author: {novice_author} (degree {degrees_dict.get(novice_author, 0)})")
"""
assert old in s, "pattern not found - tell Claude"
open("generate_results.py", "w").write(s.replace(old, new, 1))
print("patched OK")

patched OK


In [15]:
!python generate_results.py --data authors_papers.csv --format csv --model best_model.pt --scores author_scores.csv --run_gnnexplainer | tee results_log.txt

Papers: 7821
Authors (nodes): 21684
Co-authorship edges: 75779
Mean degree: 6.99
Median degree: 4.00
Max degree: 218
Isolated/degree-1 authors (novices): 2046 (9.4%)


== GNN (AR-GNN-XAI) ==
  vs h_index: {'spearman': 0.5878, 'kendall': 0.4763, 'ndcg@20': 0.1108, 'n': 21684}
  vs p_index: {'spearman': 0.9706, 'kendall': 0.8753, 'ndcg@20': 0.8522, 'n': 21684}

== degree ==
  vs h_index: {'spearman': 0.4101, 'kendall': 0.3472, 'ndcg@20': 0.1584, 'n': 21684}
  vs p_index: {'spearman': 0.3633, 'kendall': 0.2647, 'ndcg@20': 0.0043, 'n': 21684}

== closeness ==
  vs h_index: {'spearman': 0.3545, 'kendall': 0.2856, 'ndcg@20': 0.77, 'n': 21684}
  vs p_index: {'spearman': 0.3462, 'kendall': 0.2434, 'ndcg@20': 0.1097, 'n': 21684}

== NAP ==
  vs h_index: {'spearman': 0.6735, 'kendall': 0.5621, 'ndcg@20': 0.4171, 'n': 21684}
  vs p_index: {'spearman': 0.9177, 'kendall': 0.7745, 'ndcg@20': 0.6628, 'n': 21684}
Wrote results/comparison_table.csv
Selected hub author: https://openalex.org/A5060417049 

In [1]:
import torch, csv
from model import ARGNN
from explain import explain_with_gnnexplainer, FEATURE_NAMES
from data_loader import load_generic_csv, build_coauthorship_graph, build_feature_matrix, to_pyg_data
from baselines import normalized_author_position

pal, pc, apc = load_generic_csv("authors_papers.csv")
G = build_coauthorship_graph(pal)
nap = normalized_author_position(pal, pc)
authors, idx, X = build_feature_matrix(G, apc, nap)
data = to_pyg_data(G, authors, idx, X, apc)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ARGNN(in_channels=data.x.size(1), hidden_channels=32, heads=4).to(device)
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()
data = data.to(device)

for label, author in [("HUB", "https://openalex.org/A5060417049"),
                      ("NOVICE", "https://openalex.org/A5092061368")]:
    expl = explain_with_gnnexplainer(model, data, idx[author])
    imp = expl.node_mask.detach().cpu().numpy()[idx[author]]
    print(f"\n{label} ({author}, degree={G.degree(author)}):")
    for name, val in zip(FEATURE_NAMES, imp):
        print(f"  {name}: {val:.4f}")

ModuleNotFoundError: No module named 'model'

In [2]:
!ls

sample_data


In [2]:
from google.colab import files
uploaded = files.upload()

Saving ar_gnn_xai_backup.zip to ar_gnn_xai_backup.zip


In [3]:
!unzip -o ar_gnn_xai_backup.zip

Archive:  ar_gnn_xai_backup.zip
  inflating: authors_papers.csv      
  inflating: author_scores.csv       
  inflating: best_model.pt           
  inflating: training_log.csv        
  inflating: train_log.txt           
  inflating: baselines.py            
  inflating: colab_setup_all_files (2).py  
  inflating: colab_setup_all_files.py  
  inflating: data_loader.py          
  inflating: evaluate.py             
  inflating: explain.py              
  inflating: fetch_openalex.py       
  inflating: generate_results.py     
  inflating: model.py                
  inflating: train.py                


In [4]:
!pip install torch_geometric requests -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.1 MB/s eta 0:00:00


In [5]:
!python colab_setup_all_files.py

Wrote: ['baselines.py', 'data_loader.py', 'evaluate.py', 'explain.py', 'fetch_openalex.py', 'generate_results.py', 'model.py', 'train.py']


In [6]:
!python generate_results.py --data authors_papers.csv --format csv --model best_model.pt --scores author_scores.csv --run_gnnexplainer | tee results_log.txt

Papers: 7821
Authors (nodes): 21684
Co-authorship edges: 75779
Mean degree: 6.99
Median degree: 4.00
Max degree: 218
Isolated/degree-1 authors (novices): 2046 (9.4%)


== GNN (AR-GNN-XAI) ==
  vs h_index: {'spearman': 0.5878, 'kendall': 0.4763, 'ndcg@20': 0.1108, 'n': 21684}
  vs p_index: {'spearman': 0.9706, 'kendall': 0.8753, 'ndcg@20': 0.8522, 'n': 21684}

== degree ==
  vs h_index: {'spearman': 0.4101, 'kendall': 0.3472, 'ndcg@20': 0.1584, 'n': 21684}
  vs p_index: {'spearman': 0.3633, 'kendall': 0.2647, 'ndcg@20': 0.0043, 'n': 21684}

== closeness ==
  vs h_index: {'spearman': 0.3545, 'kendall': 0.2856, 'ndcg@20': 0.77, 'n': 21684}
  vs p_index: {'spearman': 0.3462, 'kendall': 0.2434, 'ndcg@20': 0.1097, 'n': 21684}

== NAP ==
  vs h_index: {'spearman': 0.6735, 'kendall': 0.5621, 'ndcg@20': 0.4171, 'n': 21684}
  vs p_index: {'spearman': 0.9177, 'kendall': 0.7745, 'ndcg@20': 0.6628, 'n': 21684}
Wrote results/comparison_table.csv

=== STAR AUTHOR: https://openalex.org/A5100349440 (gn

In [7]:
from google.colab import files
up = files.upload()

Saving colab_setup_all_files (2).py to colab_setup_all_files (2) (1).py


In [8]:
!python "$(ls -t colab_setup*.py | head -1)"

Wrote: ['baselines.py', 'data_loader.py', 'evaluate.py', 'explain.py', 'fetch_openalex.py', 'generate_results.py', 'model.py', 'train.py']


In [9]:
!grep -c "top20" generate_results.py

0


In [10]:
!rm -f generate_results.py
from google.colab import files
up = files.upload()

Saving generate_results.py to generate_results.py


In [11]:
!grep -c "top20" generate_results.py

2


In [12]:
!python generate_results.py --data authors_papers.csv --format csv --model best_model.pt --scores author_scores.csv --run_gnnexplainer | tee results_log.txt

Papers: 7821
Authors (nodes): 21684
Co-authorship edges: 75779
Mean degree: 6.99
Median degree: 4.00
Max degree: 218
Isolated/degree-1 authors (novices): 2046 (9.4%)


== GNN (AR-GNN-XAI) ==
  vs h_index: {'spearman': 0.5878, 'kendall': 0.4763, 'ndcg@20': 0.1108, 'n': 21684}
  vs p_index: {'spearman': 0.9706, 'kendall': 0.8753, 'ndcg@20': 0.8522, 'n': 21684}

== degree ==
  vs h_index: {'spearman': 0.4101, 'kendall': 0.3472, 'ndcg@20': 0.1584, 'n': 21684}
  vs p_index: {'spearman': 0.3633, 'kendall': 0.2647, 'ndcg@20': 0.0043, 'n': 21684}

== closeness ==
  vs h_index: {'spearman': 0.3545, 'kendall': 0.2856, 'ndcg@20': 0.77, 'n': 21684}
  vs p_index: {'spearman': 0.3462, 'kendall': 0.2434, 'ndcg@20': 0.1097, 'n': 21684}

== NAP ==
  vs h_index: {'spearman': 0.6735, 'kendall': 0.5621, 'ndcg@20': 0.4171, 'n': 21684}
  vs p_index: {'spearman': 0.9177, 'kendall': 0.7745, 'ndcg@20': 0.6628, 'n': 21684}
Wrote results/comparison_table.csv
Selected hub author: https://openalex.org/A5060417049 

In [13]:
!zip -r ar_gnn_xai_results.zip results results_log.txt
from google.colab import files
files.download("ar_gnn_xai_results.zip")

  adding: results/ (stored 0%)
  adding: results/gnnexplainer_star.png (deflated 15%)
  adding: results/training_curve.png (deflated 8%)
  adding: results/gnn_vs_hindex_scatter.png (deflated 12%)
  adding: results/method_comparison_bar.png (deflated 15%)
  adding: results/dataset_stats.txt (deflated 24%)
  adding: results/comparison_table.csv (deflated 48%)
  adding: results/gnnexplainer_novice.png (deflated 14%)
  adding: results/attention_explanation.txt (deflated 66%)
  adding: results_log.txt (deflated 66%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import torch, numpy as np, pandas as pd, inspect, sys
sys.path.insert(0, ".")
import generate_results as g
src = inspect.getsource(g)
print([l for l in src.splitlines() if "explain_with_gnnexplainer" in l][:3])

['        from explain import explain_with_gnnexplainer', '            expl = explain_with_gnnexplainer(model, data, author_idx)']


In [15]:
s = open("generate_results.py").read()
old = "            importance = node_mask[author_idx]\n"
new = old + "            print(f'IMPORTANCE {label}:', dict(zip(FEATURE_NAMES, [round(float(v),3) for v in importance])))\n"
assert old in s and "IMPORTANCE" not in s
open("generate_results.py","w").write(s.replace(old, new))
print("patched")

patched


In [16]:
!python generate_results.py --data authors_papers.csv --format csv --model best_model.pt --scores author_scores.csv --run_gnnexplainer | tee results_log.txt

Papers: 7821
Authors (nodes): 21684
Co-authorship edges: 75779
Mean degree: 6.99
Median degree: 4.00
Max degree: 218
Isolated/degree-1 authors (novices): 2046 (9.4%)


== GNN (AR-GNN-XAI) ==
  vs h_index: {'spearman': 0.5878, 'kendall': 0.4763, 'ndcg@20': 0.1108, 'n': 21684}
  vs p_index: {'spearman': 0.9706, 'kendall': 0.8753, 'ndcg@20': 0.8522, 'n': 21684}

== degree ==
  vs h_index: {'spearman': 0.4101, 'kendall': 0.3472, 'ndcg@20': 0.1584, 'n': 21684}
  vs p_index: {'spearman': 0.3633, 'kendall': 0.2647, 'ndcg@20': 0.0043, 'n': 21684}

== closeness ==
  vs h_index: {'spearman': 0.3545, 'kendall': 0.2856, 'ndcg@20': 0.77, 'n': 21684}
  vs p_index: {'spearman': 0.3462, 'kendall': 0.2434, 'ndcg@20': 0.1097, 'n': 21684}

== NAP ==
  vs h_index: {'spearman': 0.6735, 'kendall': 0.5621, 'ndcg@20': 0.4171, 'n': 21684}
  vs p_index: {'spearman': 0.9177, 'kendall': 0.7745, 'ndcg@20': 0.6628, 'n': 21684}
Wrote results/comparison_table.csv
Selected hub author: https://openalex.org/A5060417049 

In [17]:
!rm -f ar_gnn_xai_results.zip
!zip -r ar_gnn_xai_results.zip results results_log.txt
from google.colab import files
files.download("ar_gnn_xai_results.zip")

  adding: results/ (stored 0%)
  adding: results/gnnexplainer_star.png (deflated 15%)
  adding: results/training_curve.png (deflated 8%)
  adding: results/gnn_vs_hindex_scatter.png (deflated 12%)
  adding: results/method_comparison_bar.png (deflated 15%)
  adding: results/dataset_stats.txt (deflated 24%)
  adding: results/comparison_table.csv (deflated 48%)
  adding: results/gnnexplainer_novice.png (deflated 15%)
  adding: results/attention_explanation.txt (deflated 66%)
  adding: results_log.txt (deflated 66%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
!zip -r ar_gnn_xai_final_backup.zip authors_papers.csv author_scores.csv best_model.pt training_log.csv train_log.txt results results_log.txt *.py
from google.colab import files
files.download("ar_gnn_xai_final_backup.zip")

  adding: authors_papers.csv (deflated 86%)
  adding: author_scores.csv (deflated 78%)
  adding: best_model.pt (deflated 18%)
  adding: training_log.csv (deflated 50%)
  adding: train_log.txt (deflated 72%)
  adding: results/ (stored 0%)
  adding: results/gnnexplainer_star.png (deflated 15%)
  adding: results/training_curve.png (deflated 8%)
  adding: results/gnn_vs_hindex_scatter.png (deflated 12%)
  adding: results/method_comparison_bar.png (deflated 15%)
  adding: results/dataset_stats.txt (deflated 24%)
  adding: results/comparison_table.csv (deflated 48%)
  adding: results/gnnexplainer_novice.png (deflated 15%)
  adding: results/attention_explanation.txt (deflated 66%)
  adding: results_log.txt (deflated 66%)
  adding: baselines.py (deflated 60%)
  adding: colab_setup_all_files (2) (1).py (deflated 64%)
  adding: colab_setup_all_files (2).py (deflated 64%)
  adding: colab_setup_all_files.py (deflated 64%)
  adding: data_loader.py (deflated 63%)
  adding: evaluate.py (deflated 58%)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>